# Graph Explorer

This notebook loads `graph.ttl` from the project root and lets you explore the knowledge graph with SPARQL.

If the graph has not been built yet, run:

```bash
python3 pipeline/run.py --to build
```

The helper cells below give you:

- a reusable `run_query()` function for arbitrary SPARQL
- a `find_exercise()` helper for looking up exercise labels and URIs
- a `describe_exercise()` helper for browsing all triples about one exercise by label
- sample competency-question queries you can adapt


In [ ]:
from pathlib import Path
import html

import pyoxigraph as ox
from IPython.display import HTML, Markdown, display

PROJECT_ROOT = Path.cwd()
GRAPH_PATH = PROJECT_ROOT / "graph.ttl"
FEG = "https://placeholder.url#"

if not GRAPH_PATH.exists():
    raise FileNotFoundError(
        f"{GRAPH_PATH} does not exist. Run `python3 pipeline/run.py --to build` first."
    )

store = ox.Store()
store.load(GRAPH_PATH.read_bytes(), format=ox.RdfFormat.TURTLE)

triple_count = next(iter(store.query("SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }")))["n"].value
print(f"Loaded {GRAPH_PATH.name} with {triple_count} triples.")


In [ ]:
PREFIXES = f"""
PREFIX feg:  <{FEG}>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
""".strip()


def term_value(term):
    if term is None:
        return None
    return getattr(term, "value", str(term))


def render_rows(rows, columns, max_rows=25):
    if not rows:
        display(Markdown("_No rows returned._"))
        return

    visible = rows[:max_rows]
    head = "".join(f"<th>{html.escape(col)}</th>" for col in columns)
    body_rows = []
    for row in visible:
        body_rows.append(
            "<tr>" + "".join(
                f"<td>{html.escape('' if row.get(col) is None else str(row.get(col)))}</td>"
                for col in columns
            ) + "</tr>"
        )

    table = (
        "<table>"
        f"<thead><tr>{head}</tr></thead>"
        f"<tbody>{''.join(body_rows)}</tbody>"
        "</table>"
    )
    display(HTML(table))

    if len(rows) > max_rows:
        display(Markdown(f"_Showing first {max_rows} of {len(rows)} rows._"))


def run_query(query: str, max_rows: int = 25):
    result = store.query(query)
    columns = [str(var).lstrip("?") for var in result.variables]
    rows = []
    for solution in result:
        rows.append({col: term_value(solution[col]) for col in columns})

    print(f"{len(rows)} row(s)")
    render_rows(rows, columns, max_rows=max_rows)
    return rows


def find_exercise(name_fragment: str, limit: int = 20):
    escaped = name_fragment.lower().replace('"', '\\"')
    query = PREFIXES + f"""
SELECT ?exercise ?name
WHERE {{
    ?exercise a feg:Exercise ;
              rdfs:label ?name .
    FILTER(CONTAINS(LCASE(STR(?name)), \"{escaped}\"))
}}
ORDER BY ?name
LIMIT {limit}
"""
    return run_query(query, max_rows=limit)


def _resolve_exercise_uri_by_label(exercise_label: str):
    escaped = exercise_label.replace('"', '\\"')
    query = PREFIXES + f"""
SELECT ?exercise ?name
WHERE {{
    ?exercise a feg:Exercise ;
              rdfs:label ?name .
    FILTER(LCASE(STR(?name)) = LCASE(\"{escaped}\"))
}}
ORDER BY ?exercise
"""
    rows = run_query(query, max_rows=25)
    if not rows:
        raise ValueError(
            f"No exercise found with exact label {exercise_label!r}. Try find_exercise() first."
        )
    if len(rows) > 1:
        raise ValueError(
            f"Label {exercise_label!r} is ambiguous. Use find_exercise() to choose one exact label."
        )
    return rows[0]["exercise"]


def describe_exercise(exercise_label: str, max_rows: int = 50):
    exercise_uri = _resolve_exercise_uri_by_label(exercise_label)
    query = PREFIXES + f"""
SELECT ?predicate ?object
WHERE {{
    <{exercise_uri}> ?predicate ?object .
}}
ORDER BY ?predicate ?object
"""
    return run_query(query, max_rows=max_rows)


## Ad hoc SPARQL

Edit the query string below and rerun the cell whenever you want to inspect the graph directly.


In [ ]:
QUERY = PREFIXES + """
SELECT ?exercise ?name
WHERE {
    ?exercise a feg:Exercise ;
              rdfs:label ?name .
}
ORDER BY ?name
LIMIT 10
"""

run_query(QUERY)


## Competency Questions

These starter queries answer basic competency questions about the graph and double as templates you can adapt.


### 1. How many exercises are in the graph?


In [ ]:
exercise_count_query = PREFIXES + """
SELECT (COUNT(DISTINCT ?exercise) AS ?exerciseCount)
WHERE {
    ?exercise a feg:Exercise .
}
"""

run_query(exercise_count_query)


### 2. Which movement patterns are most represented?


In [ ]:
top_patterns_query = PREFIXES + """
SELECT ?patternLabel (COUNT(DISTINCT ?exercise) AS ?exerciseCount)
WHERE {
    ?exercise a feg:Exercise ;
              feg:movementPattern ?pattern .
    ?pattern rdfs:label ?patternLabel .
}
GROUP BY ?patternLabel
ORDER BY DESC(?exerciseCount)
LIMIT 15
"""

run_query(top_patterns_query)


### 3. Which exercises target the shoulder region as a prime mover?

This query walks down the SKOS muscle hierarchy from `feg:Shoulders`, so it will match region-, group-, and head-level muscle terms.


In [ ]:
shoulders_query = PREFIXES + """
SELECT ?name ?muscleLabel
WHERE {
    BIND(feg:Shoulders AS ?targetRegion)

    ?exercise a feg:Exercise ;
              rdfs:label ?name ;
              feg:hasInvolvement ?inv .

    ?inv feg:degree feg:PrimeMover ;
         feg:muscle ?muscle .

    ?muscle skos:broader* ?targetRegion .
    OPTIONAL { ?muscle rdfs:label ?muscleLabel }
}
ORDER BY ?name ?muscleLabel
LIMIT 25
"""

run_query(shoulders_query)


### 4. Which bodyweight anti-rotation exercises are in the graph?

This is a simple example of combining a movement pattern with an equipment filter.


In [ ]:
bodyweight_antirotation_query = PREFIXES + """
SELECT ?name
WHERE {
    ?exercise a feg:Exercise ;
              rdfs:label ?name ;
              feg:movementPattern feg:AntiRotation ;
              feg:equipment feg:Bodyweight .
}
ORDER BY ?name
LIMIT 25
"""

run_query(bodyweight_antirotation_query)


### 5. What are plausible substitutions for Dead Bug?

This example uses the current `Dead Bug` node directly in SPARQL, then the label-based helper below shows the human-friendly browse path.


In [ ]:
find_exercise("dead bug")


In [ ]:
dead_bug_substitutions_query = PREFIXES + """
SELECT ?candidate ?name (COUNT(DISTINCT ?sharedMuscle) AS ?sharedPrimeMovers)
WHERE {
    BIND(feg:ex_dead_bug AS ?source)

    ?source feg:movementPattern ?pattern ;
            feg:hasInvolvement ?srcInv .
    ?srcInv feg:degree feg:PrimeMover ;
            feg:muscle ?sharedMuscle .

    ?candidate feg:movementPattern ?pattern ;
               rdfs:label ?name ;
               feg:hasInvolvement ?candInv .
    ?candInv feg:degree feg:PrimeMover ;
             feg:muscle ?sharedMuscle .

    FILTER(?candidate != ?source)
}
GROUP BY ?candidate ?name
ORDER BY DESC(?sharedPrimeMovers) ?name
LIMIT 15
"""

run_query(dead_bug_substitutions_query)


### 6. What does the graph know about one exercise?

Use the exercise label, not the URI.


In [ ]:
describe_exercise("Dead Bug")
